# Pemrograman Tugas Akhir
M FARHAN ATHAULLOH - 121450117

DATA FILTERING

# Inisialisasi & Autentifikasi GEE

In [ ]:
import geemap
import ee

ee.Authenticate()
ee.Initialize(project='tugas-akhir-121450117')

# Connect Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install rasterio

In [ ]:
# Jalankan ini sekali di Colab untuk install GDAL
!apt install -y gdal-bin python3-gdal

# Data Filtering

## Overlap 20% Tile 48MWB

### Cek Data memiliki citra tidak layak (Piksel bernilai hitam dominan lebih dari 30%)

In [ ]:
import glob
import rasterio
import numpy as np
import os
import pandas as pd

# Path folder Multiband Sentinel-2
multiband_path = "/content/drive/MyDrive/Dataset3/Data_Patch/48MWB/MULTIBAND"
multiband_files = sorted(glob.glob(f"{multiband_path}/*.tif"))

# Inisialisasi kategori
categories = {
    "<30%": 0,
    "30-50%": 0,
    "50-70%": 0,
    ">70%": 0
}

# Proses semua file patch
for file in multiband_files:
    with rasterio.open(file) as src:
        try:
            # Ambil band 5,4,3
            img = src.read([5, 4, 3]).astype(float)

            # Ubah NaN jadi 0
            img = np.nan_to_num(img, nan=0.0)

            total_pixels = img.shape[1] * img.shape[2]

            # Hitung jumlah piksel nol (ketiga band = 0)
            zero_pixels = np.sum(np.all(img == 0, axis=0))

            # Rasio nol
            zero_ratio = zero_pixels / total_pixels

            # Masukkan ke kategori
            if zero_ratio < 0.3:
                categories["<30%"] += 1
            elif zero_ratio < 0.5:
                categories["30-50%"] += 1
            elif zero_ratio < 0.7:
                categories["50-70%"] += 1
            else:
                categories[">70%"] += 1

        except Exception as e:
            print(f" Gagal membaca {file}, error: {e}")

# Buat tabel hasil
stat_table = pd.DataFrame.from_dict(categories, orient='index', columns=['Jumlah Patch'])

print("\n Statistik Piksel 0 (Band 5,4,3):\n")
print(stat_table)

In [ ]:
import glob
import rasterio
import numpy as np
import os
import pandas as pd

# Path input dan output
multiband_path = "/content/drive/MyDrive/Dataset3/Data_Patch/48MWB/MULTIBAND"
output_path = "/content/drive/MyDrive/Dataset3/Data_Patch/48MWB/MULTIBAND_CLEAN"

# Buat folder output kalau belum ada
os.makedirs(output_path, exist_ok=True)

# Ambil semua file multiband
multiband_files = sorted(glob.glob(f"{multiband_path}/*.tif"))

# Inisialisasi kategori
categories = {
    "<30%": 0,
    "30-50%": 0,
    "50-70%": 0,
    ">70%": 0
}

# Proses semua patch
for file in multiband_files:
    with rasterio.open(file) as src:
        try:
            # Baca semua band
            img_all = src.read().astype(float)

            # Ubah NaN jadi 0
            img_all = np.nan_to_num(img_all, nan=0.0)

            # Cek band 5,4,3 saja untuk statistik nol
            img_rgb = img_all[[5-1, 4-1, 3-1], :, :]  # indeks dikurangi 1

            total_pixels = img_rgb.shape[1] * img_rgb.shape[2]
            zero_pixels = np.sum(np.all(img_rgb == 0, axis=0))
            zero_ratio = zero_pixels / total_pixels

            # Masukkan ke kategori
            if zero_ratio < 0.3:
                categories["<30%"] += 1
            elif zero_ratio < 0.5:
                categories["30-50%"] += 1
            elif zero_ratio < 0.7:
                categories["50-70%"] += 1
            else:
                categories[">70%"] += 1

            # Simpan hasil patch bersih ke folder baru
            out_file = os.path.join(output_path, os.path.basename(file))
            profile = src.profile
            with rasterio.open(out_file, "w", **profile) as dst:
                dst.write(img_all.astype(profile["dtype"]))

        except Exception as e:
            print(f" Gagal memproses {file}, error: {e}")

# Buat tabel hasil
stat_table = pd.DataFrame.from_dict(categories, orient='index', columns=['Jumlah Patch'])

print("\n Statistik Piksel 0 (Band 5,4,3):\n")
print(stat_table)

### Menampilkan citra difilter (tidak layak)

In [ ]:
import glob
import matplotlib.pyplot as plt
import rasterio
import numpy as np
import os
import pandas as pd

# Path ke folder patch Sentinel-2 tile 48MWB
base_path = "/content/drive/MyDrive/Dataset3/Data_Patch/48MWB"

multiband_files = sorted(glob.glob(f"{base_path}/MULTIBAND_CLEAN/*.tif"))
mask_files      = sorted(glob.glob(f"{base_path}/MASK/*.tif"))

# Pastikan jumlah patch sama
jumlah_patch = min(len(multiband_files), len(mask_files))
print(f"Total set patch tersedia: {jumlah_patch}\n")

# Cek >30% piksel B8,B4,B3 == 0
def is_black_dominated(path, threshold=0.3):
    with rasterio.open(path) as src:
        # Baca band 3,4,5 → B8, B4, B3
        img = src.read([3, 4, 5])
        total = img.shape[1] * img.shape[2]
        black = np.sum(np.all(img == 0, axis=0))
        return (black / total) > threshold

# Fungsi tampilkan multiband dan mask
def tampilkan_patch(multi_path, mask_path, idx):
    fig, axs = plt.subplots(1, 2, figsize=(12,5))
    name_m = os.path.basename(multi_path)
    name_k = os.path.basename(mask_path)

    # MULTIBAND (B8,B4,B3)
    with rasterio.open(multi_path) as src:
        try:
            img = src.read([3,4,5])                     # (3, H, W)
            img = np.transpose(img, (1,2,0)).astype(np.float32)
            # Min-Max normalisasi per channel
            for c in range(3):
                band = img[:,:,c]
                mn, mx = band.min(), band.max()
                img[:,:,c] = (band - mn) / (mx - mn + 1e-6)
            axs[0].imshow(img)
            axs[0].set_title(f"MULTIBAND (B8-4-3)\n{name_m}", fontsize=9)
        except Exception as e:
            axs[0].imshow(np.zeros((100,100)))
            axs[0].set_title(f"ERROR MULTIBAND\n{name_m}", fontsize=9)

    # MASK
    with rasterio.open(mask_path) as src:
        mask = src.read(1)
        axs[1].imshow(mask, cmap='gray', vmin=0, vmax=1)
        axs[1].set_title(f"MASK\n{name_k}", fontsize=9)

    for ax in axs:
        ax.axis('off')
    plt.suptitle(f"Sentinel-2 48MWB – Patch #{idx}", fontsize=12)
    plt.tight_layout()
    plt.show()

# Filter & tampilkan
filtered = []
count = 0
for m, k in zip(multiband_files, mask_files):
    if is_black_dominated(m):
        count += 1
        tampilkan_patch(m, k, count)
        filtered.append(os.path.basename(m))

# Ringkasan
df = pd.DataFrame(filtered, columns=['filename'])
print(f"\nJumlah patch >30% pixel B8,B4,B3=0: {count} / {jumlah_patch}")
print("Daftar patch:")
print(df)

### Menampilkan Citra Layak (setelahfilter)

In [ ]:
import glob
import rasterio
import numpy as np
import os
import shutil

# Path data Sentinel-2
base_path = "/content/drive/MyDrive/Dataset3/Data_Patch/48MWB"

# Path folder hasil filter
filtered_path = os.path.join(base_path, "FILTERED")
multiband_filtered = os.path.join(filtered_path, "MULTIBAND_FILTERED")
mask_filtered = os.path.join(filtered_path, "MASK_FILTERED")

# Buat folder jika belum ada
os.makedirs(multiband_filtered, exist_ok=True)
os.makedirs(mask_filtered, exist_ok=True)

# Ambil file .tif dari masing-masing folder
multiband_files = sorted(glob.glob(f"{base_path}/MULTIBAND_CLEAN/*.tif"))
mask_files = sorted(glob.glob(f"{base_path}/MASK/*.tif"))

# Fungsi untuk cek dominasi piksel hitam
def is_black_dominated(multiband_path, threshold=0.3):
    with rasterio.open(multiband_path) as src:
        try:
            img = src.read()  # Semua band
            total_pixels = img.shape[1] * img.shape[2]
            black_pixels = np.sum(np.all(img == 0, axis=0))
            black_ratio = black_pixels / total_pixels
            return black_ratio > threshold
        except:
            return True  # Gagal baca dianggap tidak lolos

# Proses filter
filtered_out = []
copied_count = 0

for m, k in zip(multiband_files, mask_files):
    filename = os.path.basename(m)

    if not is_black_dominated(m):  # Jika layak
        shutil.copy(m, os.path.join(multiband_filtered, filename))
        shutil.copy(k, os.path.join(mask_filtered, os.path.basename(k)))
        copied_count += 1
    else:
        filtered_out.append(filename)

# Tampilkan hasil
print(f"\nTotal patch tersedia: {len(multiband_files)}")
print(f"Jumlah patch >30% pixel 0 di semua band: {len(filtered_out)} / {len(multiband_files)}")
print(f"Jumlah patch LOLOS filter: {copied_count}")

# Daftar patch tidak lolos
if filtered_out:
    print("\nDaftar patch yang tidak lolos:")
    for f in filtered_out:
        print("-", f)
else:
    print("\nSemua patch lolos filter.")

# Hitung file hasil
def count_filtered(folder_path):
    return len(glob.glob(f"{folder_path}/*.tif"))

print("\nJumlah data dalam folder hasil filter:")
print(f"MULTIBAND_FILTERED:  {count_filtered(multiband_filtered)} file")
print(f"MASK_FILTERED:       {count_filtered(mask_filtered)} file")

## Overlap 20% Tile 48MVC

### Cek Data memiliki citra tidak layak (Piksel bernilai hitam dominan lebih dari 30%)

In [ ]:
import glob
import rasterio
import numpy as np
import os
import pandas as pd

# Path folder Multiband Sentinel-2
multiband_path = "/content/drive/MyDrive/Dataset3/Data_Patch/48MVC/MULTIBAND"
multiband_files = sorted(glob.glob(f"{multiband_path}/*.tif"))

# Inisialisasi kategori
categories = {
    "<30%": 0,
    "30-50%": 0,
    "50-70%": 0,
    ">70%": 0
}

# Proses semua file patch
for file in multiband_files:
    with rasterio.open(file) as src:
        try:
            # Ambil band 5,4,3
            img = src.read([5, 4, 3]).astype(float)

            # Ubah NaN jadi 0
            img = np.nan_to_num(img, nan=0.0)

            total_pixels = img.shape[1] * img.shape[2]

            # Hitung jumlah piksel nol (ketiga band = 0)
            zero_pixels = np.sum(np.all(img == 0, axis=0))

            # Rasio nol
            zero_ratio = zero_pixels / total_pixels

            # Masukkan ke kategori
            if zero_ratio < 0.3:
                categories["<30%"] += 1
            elif zero_ratio < 0.5:
                categories["30-50%"] += 1
            elif zero_ratio < 0.7:
                categories["50-70%"] += 1
            else:
                categories[">70%"] += 1

        except Exception as e:
            print(f" Gagal membaca {file}, error: {e}")

# Buat tabel hasil
stat_table = pd.DataFrame.from_dict(categories, orient='index', columns=['Jumlah Patch'])

print("\n Statistik Piksel 0 (Band 5,4,3):\n")
print(stat_table)

In [ ]:
import glob
import rasterio
import numpy as np
import os
import pandas as pd

# Path input dan output
multiband_path = "/content/drive/MyDrive/Dataset3/Data_Patch/48MVC/MULTIBAND"
output_path = "/content/drive/MyDrive/Dataset3/Data_Patch/48MVC/MULTIBAND_CLEAN"

# Buat folder output kalau belum ada
os.makedirs(output_path, exist_ok=True)

# Ambil semua file multiband
multiband_files = sorted(glob.glob(f"{multiband_path}/*.tif"))

# Inisialisasi kategori
categories = {
    "<30%": 0,
    "30-50%": 0,
    "50-70%": 0,
    ">70%": 0
}

# Proses semua patch
for file in multiband_files:
    with rasterio.open(file) as src:
        try:
            # Baca semua band
            img_all = src.read().astype(float)

            # Ubah NaN jadi 0
            img_all = np.nan_to_num(img_all, nan=0.0)

            # Cek band 5,4,3 saja untuk statistik nol
            img_rgb = img_all[[5-1, 4-1, 3-1], :, :]  # indeks dikurangi 1

            total_pixels = img_rgb.shape[1] * img_rgb.shape[2]
            zero_pixels = np.sum(np.all(img_rgb == 0, axis=0))
            zero_ratio = zero_pixels / total_pixels

            # Masukkan ke kategori
            if zero_ratio < 0.3:
                categories["<30%"] += 1
            elif zero_ratio < 0.5:
                categories["30-50%"] += 1
            elif zero_ratio < 0.7:
                categories["50-70%"] += 1
            else:
                categories[">70%"] += 1

            # Simpan hasil patch bersih ke folder baru
            out_file = os.path.join(output_path, os.path.basename(file))
            profile = src.profile
            with rasterio.open(out_file, "w", **profile) as dst:
                dst.write(img_all.astype(profile["dtype"]))

        except Exception as e:
            print(f" Gagal memproses {file}, error: {e}")

# Buat tabel hasil
stat_table = pd.DataFrame.from_dict(categories, orient='index', columns=['Jumlah Patch'])

print("\n Statistik Piksel 0 (Band 5,4,3):\n")
print(stat_table)

### Menampilkan citra difilter (tidak layak)

In [ ]:
import glob
import matplotlib.pyplot as plt
import rasterio
import numpy as np
import os
import pandas as pd

# Path ke folder patch Sentinel-2 tile 48MWB
base_path = "/content/drive/MyDrive/Dataset3/Data_Patch/48MVC"

multiband_files = sorted(glob.glob(f"{base_path}/MULTIBAND_CLEAN/*.tif"))
mask_files      = sorted(glob.glob(f"{base_path}/MASK/*.tif"))

# Pastikan jumlah patch sama
jumlah_patch = min(len(multiband_files), len(mask_files))
print(f"Total set patch tersedia: {jumlah_patch}\n")

# Cek >30% piksel B8,B4,B3 == 0
def is_black_dominated(path, threshold=0.3):
    with rasterio.open(path) as src:
        # Baca band 3,4,5 → B8, B4, B3
        img = src.read([3, 4, 5])
        total = img.shape[1] * img.shape[2]
        black = np.sum(np.all(img == 0, axis=0))
        return (black / total) > threshold

# Fungsi tampilkan multiband dan mask
def tampilkan_patch(multi_path, mask_path, idx):
    fig, axs = plt.subplots(1, 2, figsize=(12,5))
    name_m = os.path.basename(multi_path)
    name_k = os.path.basename(mask_path)

    # MULTIBAND (B8,B4,B3)
    with rasterio.open(multi_path) as src:
        try:
            img = src.read([3,4,5])                     # (3, H, W)
            img = np.transpose(img, (1,2,0)).astype(np.float32)
            # Min-Max normalisasi per channel
            for c in range(3):
                band = img[:,:,c]
                mn, mx = band.min(), band.max()
                img[:,:,c] = (band - mn) / (mx - mn + 1e-6)
            axs[0].imshow(img)
            axs[0].set_title(f"MULTIBAND (B8-4-3)\n{name_m}", fontsize=9)
        except Exception as e:
            axs[0].imshow(np.zeros((100,100)))
            axs[0].set_title(f"ERROR MULTIBAND\n{name_m}", fontsize=9)

    # MASK
    with rasterio.open(mask_path) as src:
        mask = src.read(1)
        axs[1].imshow(mask, cmap='gray', vmin=0, vmax=1)
        axs[1].set_title(f"MASK\n{name_k}", fontsize=9)

    for ax in axs:
        ax.axis('off')
    plt.suptitle(f"Sentinel-2 48MWB – Patch #{idx}", fontsize=12)
    plt.tight_layout()
    plt.show()

# Filter & tampilkan
filtered = []
count = 0
for m, k in zip(multiband_files, mask_files):
    if is_black_dominated(m):
        count += 1
        tampilkan_patch(m, k, count)
        filtered.append(os.path.basename(m))

# Ringkasan
df = pd.DataFrame(filtered, columns=['filename'])
print(f"\nJumlah patch >30% pixel B8,B4,B3=0: {count} / {jumlah_patch}")
print("Daftar patch:")
print(df)

### Menampilkan Citra Layak (setelahfilter)

In [ ]:
import glob
import rasterio
import numpy as np
import os
import shutil

# Path data Sentinel-2
base_path = "/content/drive/MyDrive/Dataset3/Data_Patch/48MVC"

# Path folder hasil filter
filtered_path = os.path.join(base_path, "FILTERED")
multiband_filtered = os.path.join(filtered_path, "MULTIBAND_FILTERED")
mask_filtered = os.path.join(filtered_path, "MASK_FILTERED")

# Buat folder jika belum ada
os.makedirs(multiband_filtered, exist_ok=True)
os.makedirs(mask_filtered, exist_ok=True)

# Ambil file .tif dari masing-masing folder
multiband_files = sorted(glob.glob(f"{base_path}/MULTIBAND_CLEAN/*.tif"))
mask_files = sorted(glob.glob(f"{base_path}/MASK/*.tif"))

# Fungsi untuk cek dominasi piksel hitam
def is_black_dominated(multiband_path, threshold=0.3):
    with rasterio.open(multiband_path) as src:
        try:
            img = src.read()  # Semua band
            total_pixels = img.shape[1] * img.shape[2]
            black_pixels = np.sum(np.all(img == 0, axis=0))
            black_ratio = black_pixels / total_pixels
            return black_ratio > threshold
        except:
            return True  # Gagal baca dianggap tidak lolos

# Proses filter
filtered_out = []
copied_count = 0

for m, k in zip(multiband_files, mask_files):
    filename = os.path.basename(m)

    if not is_black_dominated(m):  # Jika layak
        shutil.copy(m, os.path.join(multiband_filtered, filename))
        shutil.copy(k, os.path.join(mask_filtered, os.path.basename(k)))
        copied_count += 1
    else:
        filtered_out.append(filename)

# Tampilkan hasil
print(f"\nTotal patch tersedia: {len(multiband_files)}")
print(f"Jumlah patch >30% pixel 0 di semua band: {len(filtered_out)} / {len(multiband_files)}")
print(f"Jumlah patch LOLOS filter: {copied_count}")

# Daftar patch tidak lolos
if filtered_out:
    print("\nDaftar patch yang tidak lolos:")
    for f in filtered_out:
        print("-", f)
else:
    print("\nSemua patch lolos filter.")

# Hitung file hasil
def count_filtered(folder_path):
    return len(glob.glob(f"{folder_path}/*.tif"))

print("\nJumlah data dalam folder hasil filter:")
print(f"MULTIBAND_FILTERED:  {count_filtered(multiband_filtered)} file")
print(f"MASK_FILTERED:       {count_filtered(mask_filtered)} file")

## Overlap 20% Tile 48MUC

### Cek Data memiliki citra tidak layak (Piksel bernilai hitam dominan lebih dari 30%)

In [ ]:
import glob
import rasterio
import numpy as np
import os
import pandas as pd

# Path folder Multiband Sentinel-2
multiband_path = "/content/drive/MyDrive/Dataset3/Data_Patch/48MUC/MULTIBAND"
multiband_files = sorted(glob.glob(f"{multiband_path}/*.tif"))

# Inisialisasi kategori
categories = {
    "<30%": 0,
    "30-50%": 0,
    "50-70%": 0,
    ">70%": 0
}

# Proses semua file patch
for file in multiband_files:
    with rasterio.open(file) as src:
        try:
            # Ambil band 5,4,3
            img = src.read([5, 4, 3]).astype(float)

            # Ubah NaN jadi 0
            img = np.nan_to_num(img, nan=0.0)

            total_pixels = img.shape[1] * img.shape[2]

            # Hitung jumlah piksel nol (ketiga band = 0)
            zero_pixels = np.sum(np.all(img == 0, axis=0))

            # Rasio nol
            zero_ratio = zero_pixels / total_pixels

            # Masukkan ke kategori
            if zero_ratio < 0.3:
                categories["<30%"] += 1
            elif zero_ratio < 0.5:
                categories["30-50%"] += 1
            elif zero_ratio < 0.7:
                categories["50-70%"] += 1
            else:
                categories[">70%"] += 1

        except Exception as e:
            print(f" Gagal membaca {file}, error: {e}")

# Buat tabel hasil
stat_table = pd.DataFrame.from_dict(categories, orient='index', columns=['Jumlah Patch'])

print("\n Statistik Piksel 0 (Band 5,4,3):\n")
print(stat_table)

In [ ]:
import glob
import rasterio
import numpy as np
import os
import pandas as pd

# Path input dan output
multiband_path = "/content/drive/MyDrive/Dataset3/Data_Patch/48MUC/MULTIBAND"
output_path = "/content/drive/MyDrive/Dataset3/Data_Patch/48MUC/MULTIBAND_CLEAN"

# Buat folder output kalau belum ada
os.makedirs(output_path, exist_ok=True)

# Ambil semua file multiband
multiband_files = sorted(glob.glob(f"{multiband_path}/*.tif"))

# Inisialisasi kategori
categories = {
    "<30%": 0,
    "30-50%": 0,
    "50-70%": 0,
    ">70%": 0
}

# Proses semua patch
for file in multiband_files:
    with rasterio.open(file) as src:
        try:
            # Baca semua band
            img_all = src.read().astype(float)

            # Ubah NaN jadi 0
            img_all = np.nan_to_num(img_all, nan=0.0)

            # Cek band 5,4,3 saja untuk statistik nol
            img_rgb = img_all[[5-1, 4-1, 3-1], :, :]  # indeks dikurangi 1

            total_pixels = img_rgb.shape[1] * img_rgb.shape[2]
            zero_pixels = np.sum(np.all(img_rgb == 0, axis=0))
            zero_ratio = zero_pixels / total_pixels

            # Masukkan ke kategori
            if zero_ratio < 0.3:
                categories["<30%"] += 1
            elif zero_ratio < 0.5:
                categories["30-50%"] += 1
            elif zero_ratio < 0.7:
                categories["50-70%"] += 1
            else:
                categories[">70%"] += 1

            # Simpan hasil patch bersih ke folder baru
            out_file = os.path.join(output_path, os.path.basename(file))
            profile = src.profile
            with rasterio.open(out_file, "w", **profile) as dst:
                dst.write(img_all.astype(profile["dtype"]))

        except Exception as e:
            print(f" Gagal memproses {file}, error: {e}")

# Buat tabel hasil
stat_table = pd.DataFrame.from_dict(categories, orient='index', columns=['Jumlah Patch'])

print("\n Statistik Piksel 0 (Band 5,4,3):\n")
print(stat_table)

### Menampilkan citra difilter (tidak layak)

In [ ]:
import glob
import matplotlib.pyplot as plt
import rasterio
import numpy as np
import os
import pandas as pd

# Path ke folder patch Sentinel-2 tile 48MWB
base_path = "/content/drive/MyDrive/Dataset3/Data_Patch/48MUC"

multiband_files = sorted(glob.glob(f"{base_path}/MULTIBAND_CLEAN/*.tif"))
mask_files      = sorted(glob.glob(f"{base_path}/MASK/*.tif"))

# Pastikan jumlah patch sama
jumlah_patch = min(len(multiband_files), len(mask_files))
print(f"Total set patch tersedia: {jumlah_patch}\n")

# Cek >30% piksel B8,B4,B3 == 0
def is_black_dominated(path, threshold=0.3):
    with rasterio.open(path) as src:
        # Baca band 3,4,5 → B8, B4, B3
        img = src.read([3, 4, 5])
        total = img.shape[1] * img.shape[2]
        black = np.sum(np.all(img == 0, axis=0))
        return (black / total) > threshold

# Fungsi tampilkan multiband dan mask
def tampilkan_patch(multi_path, mask_path, idx):
    fig, axs = plt.subplots(1, 2, figsize=(12,5))
    name_m = os.path.basename(multi_path)
    name_k = os.path.basename(mask_path)

    # MULTIBAND (B8,B4,B3)
    with rasterio.open(multi_path) as src:
        try:
            img = src.read([3,4,5])                     # (3, H, W)
            img = np.transpose(img, (1,2,0)).astype(np.float32)
            # Min-Max normalisasi per channel
            for c in range(3):
                band = img[:,:,c]
                mn, mx = band.min(), band.max()
                img[:,:,c] = (band - mn) / (mx - mn + 1e-6)
            axs[0].imshow(img)
            axs[0].set_title(f"MULTIBAND (B8-4-3)\n{name_m}", fontsize=9)
        except Exception as e:
            axs[0].imshow(np.zeros((100,100)))
            axs[0].set_title(f"ERROR MULTIBAND\n{name_m}", fontsize=9)

    # MASK
    with rasterio.open(mask_path) as src:
        mask = src.read(1)
        axs[1].imshow(mask, cmap='gray', vmin=0, vmax=1)
        axs[1].set_title(f"MASK\n{name_k}", fontsize=9)

    for ax in axs:
        ax.axis('off')
    plt.suptitle(f"Sentinel-2 48MWB – Patch #{idx}", fontsize=12)
    plt.tight_layout()
    plt.show()

# Filter & tampilkan
filtered = []
count = 0
for m, k in zip(multiband_files, mask_files):
    if is_black_dominated(m):
        count += 1
        tampilkan_patch(m, k, count)
        filtered.append(os.path.basename(m))

# Ringkasan
df = pd.DataFrame(filtered, columns=['filename'])
print(f"\nJumlah patch >30% pixel B8,B4,B3=0: {count} / {jumlah_patch}")
print("Daftar patch:")
print(df)

### Menampilkan Citra Layak (setelahfilter)

In [ ]:
import glob
import rasterio
import numpy as np
import os
import shutil

# Path data Sentinel-2
base_path = "/content/drive/MyDrive/Dataset3/Data_Patch/48MUC"

# Path folder hasil filter
filtered_path = os.path.join(base_path, "FILTERED")
multiband_filtered = os.path.join(filtered_path, "MULTIBAND_FILTERED")
mask_filtered = os.path.join(filtered_path, "MASK_FILTERED")

# Buat folder jika belum ada
os.makedirs(multiband_filtered, exist_ok=True)
os.makedirs(mask_filtered, exist_ok=True)

# Ambil file .tif dari masing-masing folder
multiband_files = sorted(glob.glob(f"{base_path}/MULTIBAND_CLEAN/*.tif"))
mask_files = sorted(glob.glob(f"{base_path}/MASK/*.tif"))

# Fungsi untuk cek dominasi piksel hitam
def is_black_dominated(multiband_path, threshold=0.3):
    with rasterio.open(multiband_path) as src:
        try:
            img = src.read()  # Semua band
            total_pixels = img.shape[1] * img.shape[2]
            black_pixels = np.sum(np.all(img == 0, axis=0))
            black_ratio = black_pixels / total_pixels
            return black_ratio > threshold
        except:
            return True  # Gagal baca dianggap tidak lolos

# Proses filter
filtered_out = []
copied_count = 0

for m, k in zip(multiband_files, mask_files):
    filename = os.path.basename(m)

    if not is_black_dominated(m):  # Jika layak
        shutil.copy(m, os.path.join(multiband_filtered, filename))
        shutil.copy(k, os.path.join(mask_filtered, os.path.basename(k)))
        copied_count += 1
    else:
        filtered_out.append(filename)

# Tampilkan hasil
print(f"\nTotal patch tersedia: {len(multiband_files)}")
print(f"Jumlah patch >30% pixel 0 di semua band: {len(filtered_out)} / {len(multiband_files)}")
print(f"Jumlah patch LOLOS filter: {copied_count}")

# Daftar patch tidak lolos
if filtered_out:
    print("\nDaftar patch yang tidak lolos:")
    for f in filtered_out:
        print("-", f)
else:
    print("\nSemua patch lolos filter.")

# Hitung file hasil
def count_filtered(folder_path):
    return len(glob.glob(f"{folder_path}/*.tif"))

print("\nJumlah data dalam folder hasil filter:")
print(f"MULTIBAND_FILTERED:  {count_filtered(multiband_filtered)} file")
print(f"MASK_FILTERED:       {count_filtered(mask_filtered)} file")

# Filtering Mask


# 48MWB

In [ ]:
import glob
import matplotlib.pyplot as plt
import rasterio
import numpy as np
import os
import pandas as pd

# Path ke folder patch Sentinel-2 tile 48MWB
base_path = "/content/drive/MyDrive/Dataset3/Data_Patch/48MWB"

multiband_files = sorted(glob.glob(f"{base_path}/MULTIBAND_CLEAN/*.tif"))
mask_files      = sorted(glob.glob(f"{base_path}/MASK/*.tif"))

# Pastikan jumlah patch sama
jumlah_patch = min(len(multiband_files), len(mask_files))
print(f"Total set patch tersedia: {jumlah_patch}\n")

# Filter pakai mask
def is_mask_dominated(mask_path, threshold=0.3, target_value=1):
    """
    Cek apakah patch mask memiliki >threshold proporsi piksel dengan nilai target_value.
    target_value = 1 -> cek objek
    target_value = 0 -> cek background
    """
    with rasterio.open(mask_path) as src:
        mask = src.read(1)
        total = mask.size
        count = np.sum(mask == target_value)
        return (count / total) > threshold

# Fungsi tampilkan multiband dan mask
def tampilkan_patch(multi_path, mask_path, idx):
    fig, axs = plt.subplots(1, 2, figsize=(12,5))
    name_m = os.path.basename(multi_path)
    name_k = os.path.basename(mask_path)

    # MULTIBAND (B8,B4,B3)
    with rasterio.open(multi_path) as src:
        try:
            img = src.read([3,4,5])                     # (3, H, W)
            img = np.transpose(img, (1,2,0)).astype(np.float32)
            # Min-Max normalisasi per channel
            for c in range(3):
                band = img[:,:,c]
                mn, mx = band.min(), band.max()
                img[:,:,c] = (band - mn) / (mx - mn + 1e-6)
            axs[0].imshow(img)
            axs[0].set_title(f"MULTIBAND (B8-4-3)\n{name_m}", fontsize=9)
        except Exception as e:
            axs[0].imshow(np.zeros((100,100)))
            axs[0].set_title(f"ERROR MULTIBAND\n{name_m}", fontsize=9)

    # MASK
    with rasterio.open(mask_path) as src:
        mask = src.read(1)
        axs[1].imshow(mask, cmap='gray', vmin=0, vmax=1)
        axs[1].set_title(f"MASK\n{name_k}", fontsize=9)

    for ax in axs:
        ax.axis('off')
    plt.suptitle(f"Sentinel-2 48MWB – Patch #{idx}", fontsize=12)
    plt.tight_layout()
    plt.show()

# Filter & tampilkan
filtered = []
count = 0
for m, k in zip(multiband_files, mask_files):
    if is_mask_dominated(k, threshold=0.3, target_value=1):  # ubah target_value=0 kalau mau cek background
        count += 1
        tampilkan_patch(m, k, count)
        filtered.append(os.path.basename(k))

# Ringkasan
df = pd.DataFrame(filtered, columns=['filename'])
print(f"\nJumlah patch mask dengan >30% pixel =1: {count} / {jumlah_patch}")
print("Daftar patch:")
print(df)

# 48MVC

In [ ]:
import glob
import matplotlib.pyplot as plt
import rasterio
import numpy as np
import os
import pandas as pd

# Path ke folder patch Sentinel-2 tile 48MWB
base_path = "/content/drive/MyDrive/Dataset3/Data_Patch/48MVC"

multiband_files = sorted(glob.glob(f"{base_path}/MULTIBAND_CLEAN/*.tif"))
mask_files      = sorted(glob.glob(f"{base_path}/MASK/*.tif"))

# Pastikan jumlah patch sama
jumlah_patch = min(len(multiband_files), len(mask_files))
print(f"Total set patch tersedia: {jumlah_patch}\n")

# Filter pakai mask
def is_mask_dominated(mask_path, threshold=0.3, target_value=1):
    """
    Cek apakah patch mask memiliki >threshold proporsi piksel dengan nilai target_value.
    target_value = 1 -> cek objek
    target_value = 0 -> cek background
    """
    with rasterio.open(mask_path) as src:
        mask = src.read(1)
        total = mask.size
        count = np.sum(mask == target_value)
        return (count / total) > threshold

# Fungsi tampilkan multiband dan mask
def tampilkan_patch(multi_path, mask_path, idx):
    fig, axs = plt.subplots(1, 2, figsize=(12,5))
    name_m = os.path.basename(multi_path)
    name_k = os.path.basename(mask_path)

    # MULTIBAND (B8,B4,B3)
    with rasterio.open(multi_path) as src:
        try:
            img = src.read([3,4,5])                     # (3, H, W)
            img = np.transpose(img, (1,2,0)).astype(np.float32)
            # Min-Max normalisasi per channel
            for c in range(3):
                band = img[:,:,c]
                mn, mx = band.min(), band.max()
                img[:,:,c] = (band - mn) / (mx - mn + 1e-6)
            axs[0].imshow(img)
            axs[0].set_title(f"MULTIBAND (B8-4-3)\n{name_m}", fontsize=9)
        except Exception as e:
            axs[0].imshow(np.zeros((100,100)))
            axs[0].set_title(f"ERROR MULTIBAND\n{name_m}", fontsize=9)

    # MASK
    with rasterio.open(mask_path) as src:
        mask = src.read(1)
        axs[1].imshow(mask, cmap='gray', vmin=0, vmax=1)
        axs[1].set_title(f"MASK\n{name_k}", fontsize=9)

    for ax in axs:
        ax.axis('off')
    plt.suptitle(f"Sentinel-2 48MVC – Patch #{idx}", fontsize=12)
    plt.tight_layout()
    plt.show()

# Filter & tampilkan
filtered = []
count = 0
for m, k in zip(multiband_files, mask_files):
    if is_mask_dominated(k, threshold=0.3, target_value=1):  # ubah target_value=0 kalau mau cek background
        count += 1
        tampilkan_patch(m, k, count)
        filtered.append(os.path.basename(k))

# Ringkasan
df = pd.DataFrame(filtered, columns=['filename'])
print(f"\nJumlah patch mask dengan >30% pixel =1: {count} / {jumlah_patch}")
print("Daftar patch:")
print(df)

#48MUC

In [ ]:
import glob
import matplotlib.pyplot as plt
import rasterio
import numpy as np
import os
import pandas as pd

# Path ke folder patch Sentinel-2 tile 48MWB
base_path = "/content/drive/MyDrive/Dataset3/Data_Patch/48MUC"

multiband_files = sorted(glob.glob(f"{base_path}/MULTIBAND_CLEAN/*.tif"))
mask_files      = sorted(glob.glob(f"{base_path}/MASK/*.tif"))

# Pastikan jumlah patch sama
jumlah_patch = min(len(multiband_files), len(mask_files))
print(f"Total set patch tersedia: {jumlah_patch}\n")

# Filter pakai mask
def is_mask_dominated(mask_path, threshold=0.3, target_value=1):
    """
    Cek apakah patch mask memiliki >threshold proporsi piksel dengan nilai target_value.
    target_value = 1 -> cek objek
    target_value = 0 -> cek background
    """
    with rasterio.open(mask_path) as src:
        mask = src.read(1)
        total = mask.size
        count = np.sum(mask == target_value)
        return (count / total) > threshold

# Fungsi tampilkan multiband dan mask
def tampilkan_patch(multi_path, mask_path, idx):
    fig, axs = plt.subplots(1, 2, figsize=(12,5))
    name_m = os.path.basename(multi_path)
    name_k = os.path.basename(mask_path)

    # MULTIBAND (B8,B4,B3)
    with rasterio.open(multi_path) as src:
        try:
            img = src.read([3,4,5])                     # (3, H, W)
            img = np.transpose(img, (1,2,0)).astype(np.float32)
            # Min-Max normalisasi per channel
            for c in range(3):
                band = img[:,:,c]
                mn, mx = band.min(), band.max()
                img[:,:,c] = (band - mn) / (mx - mn + 1e-6)
            axs[0].imshow(img)
            axs[0].set_title(f"MULTIBAND (B8-4-3)\n{name_m}", fontsize=9)
        except Exception as e:
            axs[0].imshow(np.zeros((100,100)))
            axs[0].set_title(f"ERROR MULTIBAND\n{name_m}", fontsize=9)

    # MASK
    with rasterio.open(mask_path) as src:
        mask = src.read(1)
        axs[1].imshow(mask, cmap='gray', vmin=0, vmax=1)
        axs[1].set_title(f"MASK\n{name_k}", fontsize=9)

    for ax in axs:
        ax.axis('off')
    plt.suptitle(f"Sentinel-2 48MUC – Patch #{idx}", fontsize=12)
    plt.tight_layout()
    plt.show()

# Filter & tampilkan
filtered = []
count = 0
for m, k in zip(multiband_files, mask_files):
    if is_mask_dominated(k, threshold=0.3, target_value=1):  # ubah target_value=0 kalau mau cek background
        count += 1
        tampilkan_patch(m, k, count)
        filtered.append(os.path.basename(k))

# Ringkasan
df = pd.DataFrame(filtered, columns=['filename'])
print(f"\nJumlah patch mask dengan >30% pixel =1: {count} / {jumlah_patch}")
print("Daftar patch:")
print(df)

# Menampilkan Semua Citra FILTERED

In [ ]:
import glob
import os
import rasterio
import numpy as np
import matplotlib.pyplot as plt

# Path
base_path = "/content/drive/MyDrive/Dataset3/Data_Patch/48MUC/FILTERED"
multiband_path = os.path.join(base_path, "MULTIBAND_FILTERED")
mask_path      = os.path.join(base_path, "MASK_FILTERED")

# Ambil file dan sort biar sesuai
multiband_files = sorted(glob.glob(f"{multiband_path}/*.tif"))
mask_files      = sorted(glob.glob(f"{mask_path}/*.tif"))

print(f"Total pasangan patch: {len(multiband_files)}")

# Tampilkan pasangan citra
for m_file, k_file in zip(multiband_files, mask_files):
    fig, axs = plt.subplots(1, 2, figsize=(12, 6))

    # MULTIBAND (Band 5,4,3)
    with rasterio.open(m_file) as src:
        img = src.read([5, 4, 3]).astype(float)  # (3,H,W)
        img = np.transpose(img, (1, 2, 0))      # (H,W,3)

        # Normalisasi min-max biar tampil bagus
        for c in range(3):
            band = img[:,:,c]
            mn, mx = np.min(band), np.max(band)
            if mx > mn:  # hindari pembagian nol
                img[:,:,c] = (band - mn) / (mx - mn)

        axs[0].imshow(img)
        axs[0].set_title(f"Multiband (B5,B4,B3)\n{os.path.basename(m_file)}", fontsize=9)
        axs[0].axis('off')

    # MASK (Band 1)
    with rasterio.open(k_file) as src:
        mask = src.read(1)
        axs[1].imshow(mask, cmap='gray', vmin=0, vmax=1)
        axs[1].set_title(f"Mask\n{os.path.basename(k_file)}", fontsize=9)
        axs[1].axis('off')

    plt.tight_layout()
    plt.show()


# Mengabungkan Folder Menjadi 1 Folder

In [ ]:
import os
import shutil

def gabungkan_folder(folder_sumber, folder_tujuan):
    """
    Menyalin semua file dari folder sumber ke folder tujuan.
    """
    os.makedirs(folder_tujuan, exist_ok=True)
    for root, _, files in os.walk(folder_sumber):
        for file in files:
            sumber = os.path.join(root, file)
            tujuan = os.path.join(folder_tujuan, file)
            shutil.copy2(sumber, tujuan)  # Menyalin file beserta metadata

# Direktori dasar sumber Sentinel-2
base_path = "/content/drive/MyDrive/Dataset3/Data_Patch/"
folder_48MUC = os.path.join(base_path, "48MUC/FILTERED")
folder_48MWB = os.path.join(base_path, "48MWB/FILTERED")
folder_48MVC = os.path.join(base_path, "48MVC/FILTERED")

# Folder sumber per jenis data
mask_48MUC = os.path.join(folder_48MUC, "MASK_FILTERED")
mask_48MWB = os.path.join(folder_48MWB, "MASK_FILTERED")
mask_48MVC = os.path.join(folder_48MVC, "MASK_FILTERED")

mb_48MUC = os.path.join(folder_48MUC, "MULTIBAND_FILTERED")
mb_48MWB = os.path.join(folder_48MWB, "MULTIBAND_FILTERED")
mb_48MVC = os.path.join(folder_48MVC, "MULTIBAND_FILTERED")

# Folder tujuan gabungan
tujuan_base = "/content/drive/MyDrive/Dataset3/Data_Burned_Sentinel-2"
mask_tujuan = os.path.join(tujuan_base, "MASK")
multiband_tujuan = os.path.join(tujuan_base, "MULTIBAND")

# Proses penggabungan
gabungkan_folder(mask_48MUC, mask_tujuan)
gabungkan_folder(mask_48MWB, mask_tujuan)
gabungkan_folder(mask_48MVC, mask_tujuan)

gabungkan_folder(mb_48MUC, multiband_tujuan)
gabungkan_folder(mb_48MWB, multiband_tujuan)
gabungkan_folder(mb_48MVC, multiband_tujuan)

print(" Penggabungan selesai untuk MASK dan MULTIBAND Sentinel-2.")

In [ ]:
# Tampilkan jumlah file hasil penggabungan
def hitung_jumlah_file(folder):
    """
    Menghitung jumlah file di dalam suatu folder (termasuk subfolder).
    """
    return sum(len(files) for _, _, files in os.walk(folder))

jumlah_mask = hitung_jumlah_file(mask_tujuan)
jumlah_mb = hitung_jumlah_file(multiband_tujuan)

print(f"Total citra MASK       : {jumlah_mask}")
print(f"Total citra MULTIBAND  : {jumlah_mb}")